In [4]:
import pandas as pd
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch
import json
import os
import time
from tqdm import tqdm

In [5]:
# Đường dẫn đến file Excel (thay đổi theo file của bạn)
excel_file = 'Confessions of HNMU.xlsx'

In [ ]:
# Đường dẫn đến thư mục lưu trữ tùy chỉnh (thay đổi theo ổ cứng/thư mục của bạn)
custom_cache_dir = "D:/Model"

# Đảm bảo thư mục tồn tại, nếu không thì tạo mới
os.makedirs(custom_cache_dir, exist_ok=True)

# Tên mô hình trên HuggingFace
model_name = "Viet-Mistral/Vistral-7B-Chat"

# Tải tokenizer và lưu vào thư mục tùy chỉnh
tokenizer = AutoTokenizer.from_pretrained(
    model_name,
    cache_dir=custom_cache_dir,
    local_files_only=False  # Tải từ HuggingFace nếu chưa có
)

# Tải mô hình và lưu vào thư mục tùy chỉnh
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    cache_dir=custom_cache_dir,
    torch_dtype=torch.float16,  # Sử dụng float16 để tiết kiệm bộ nhớ
    device_map="auto",  # Tự động phân bổ thiết bị (CPU/GPU)
    local_files_only=False  # Tải từ HuggingFace nếu chưa có
)

print(f"Mô hình và tokenizer đã được tải và lưu vào: {custom_cache_dir}")

'(ReadTimeoutError("HTTPSConnectionPool(host='huggingface.co', port=443): Read timed out. (read timeout=10)"), '(Request ID: 9fcd1afc-4697-4572-aa16-378051945e60)')' thrown while requesting HEAD https://huggingface.co/Viet-Mistral/Vistral-7B-Chat/resolve/main/tokenizer_config.json
Retrying in 1s [Retry 1/5].


tokenizer_config.json:   0%|          | 0.00/2.52k [00:00<?, ?B/s]

tokenizer.model:   0%|          | 0.00/597k [00:00<?, ?B/s]

In [ ]:
# Hàm để kiểm tra và trích xuất từ toxic sử dụng mô hình
def extract_toxic_words(text):
    # Prompt để yêu cầu mô hình trích xuất từ toxic
    prompt = f"""
    Phân tích văn bản sau và trích xuất các từ mang tính toxic, lăng mạ, tục tĩu. Chỉ trả về danh sách các từ đó, nếu không có thì trả về 'None'.
    Văn bản: {text}
    """
    
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    outputs = model.generate(**inputs, max_new_tokens=100, temperature=0.7)
    response = tokenizer.decode(outputs[0], skip_special_tokens=True)
    
    # Xử lý phản hồi để lấy kết quả
    result = response.strip().split('\n')[-1]  # Giả định phản hồi cuối là kết quả
    if 'None' in result:
        return None
    else:
        # Giả định kết quả là danh sách từ, ví dụ: "từ1, từ2"
        return result.split(', ')

In [ ]:
# Đọc file Excel
df = pd.read_excel(excel_file)

# Giả định cột chứa văn bản là 'text' (thay đổi nếu khác)
text_column = 'post_text'  # Thay đổi tên cột nếu cần

# Lọc và trích xuất toxic words từ từng hàng
toxic_results = {}
for idx, row in df.iterrows():
    text = row[text_column]
    toxic_words = extract_toxic_words(text)
    if toxic_words:
        toxic_results[idx] = toxic_words

In [ ]:
# Kết quả kiểm tra toxic
toxic_results = []  # Ví dụ danh sách rỗng

# Chuẩn bị output JSON
if toxic_results:
    output = {"toxic_results": toxic_results}
else:
    output = {"toxic_results": "None"}

# Đường dẫn thư mục và file
folder_path = "output"
file_path = os.path.join(folder_path, "toxic_results.json")

# Các bước xử lý
steps = [
    "Chuẩn bị dữ liệu JSON",
    "Kiểm tra & tạo thư mục",
    "Ghi dữ liệu vào file",
    "Hoàn tất & in kết quả"
]

# Progress bar chi tiết với ETA
with tqdm(total=len(steps), desc="Tiến trình", unit="step", ncols=100,
          bar_format="{l_bar}{bar} | {n_fmt}/{total_fmt} [{elapsed}<{remaining}, {percentage:3.0f}%]") as pbar:

    # 1. Chuẩn bị dữ liệu
    time.sleep(0.5)
    pbar.set_postfix_str(steps[0])
    pbar.update(1)

    # 2. Kiểm tra & tạo thư mục
    os.makedirs(folder_path, exist_ok=True)
    time.sleep(0.5)
    pbar.set_postfix_str(steps[1])
    pbar.update(1)

    # 3. Ghi dữ liệu vào file
    with open(file_path, "w", encoding="utf-8") as f:
        json.dump(output, f, ensure_ascii=False, indent=4)
    time.sleep(0.5)
    pbar.set_postfix_str(steps[2])
    pbar.update(1)

    # 4. Hoàn tất
    print("\nOutput JSON:")
    print(json.dumps(output, ensure_ascii=False, indent=4))
    print(f"✔ Kết quả đã được lưu vào: {file_path}")
    time.sleep(0.5)
    pbar.set_postfix_str(steps[3])
    pbar.update(1)


In [ ]:
if __name__ == "__main__":
    pass